# SozKZ MoE 3B Training

Training a Mixtral-style MoE model (128 experts, top-2, shared router) on ~17.9B Kazakh tokens.

- **Model**: `stukenov/sozkz-moe-mix-3b-kk-base-v1-init` (~3B params, ~180M active)
- **Datasets**: 2 pre-tokenized datasets (~17.36M blocks x 1024 tokens)
- **Tokenizer**: `stukenov/sozkz-core-gpt2-50k-kk-base-v1`
- **GPU**: Tesla A100 40/80GB

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers datasets accelerate huggingface_hub

## 2. Check GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}, {props.total_mem / 1e9:.1f} GB")

## 3. Login to HuggingFace (for model download & upload)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Load Datasets

In [ ]:
from datasets import load_dataset, concatenate_datasets, Features, Sequence, Value

print("Loading dataset 1: sozkz-corpus-tokenized-kk-llama50k-v3...")
ds1 = load_dataset("stukenov/sozkz-corpus-tokenized-kk-llama50k-v3")
print(f"  Train: {len(ds1['train']):,}, Val: {len(ds1['validation']):,}")

print("Loading dataset 2: sozkz-corpus-tokenized-enkk-fineweb-edu-v1...")
ds2 = load_dataset("stukenov/sozkz-corpus-tokenized-enkk-fineweb-edu-v1")
print(f"  Train: {len(ds2['train']):,}, Val: {len(ds2['validation']):,}")

# Unify columns: keep only input_ids and labels
cols_to_keep = ["input_ids", "labels"]
ds1_train = ds1["train"].remove_columns([c for c in ds1["train"].column_names if c not in cols_to_keep])
ds1_val = ds1["validation"].remove_columns([c for c in ds1["validation"].column_names if c not in cols_to_keep])
ds2_train = ds2["train"].remove_columns([c for c in ds2["train"].column_names if c not in cols_to_keep])
ds2_val = ds2["validation"].remove_columns([c for c in ds2["validation"].column_names if c not in cols_to_keep])

# Cast ds2 labels from int64 to int32 to match ds1
target_features = Features({
    "input_ids": Sequence(Value("int32")),
    "labels": Sequence(Value("int32")),
})
ds2_train = ds2_train.cast(target_features)
ds2_val = ds2_val.cast(target_features)

train_dataset = concatenate_datasets([ds1_train, ds2_train])
val_dataset = concatenate_datasets([ds1_val, ds2_val])

print(f"\nCombined Train: {len(train_dataset):,}")
print(f"Combined Val: {len(val_dataset):,}")
print(f"Total tokens: ~{len(train_dataset) * 1024 / 1e9:.1f}B")

## 5. Load Model & Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "stukenov/sozkz-moe-mix-3b-kk-base-v1-init"
TOKENIZER_NAME = "stukenov/sozkz-core-gpt2-50k-kk-base-v1"

print(f"Loading tokenizer: {TOKENIZER_NAME}")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
)

total_params = sum(p.numel() for p in model.parameters())
unique_params = sum(p.numel() for p in set(model.parameters()))
print(f"Total params (with sharing): {total_params / 1e9:.2f}B")
print(f"Unique params: {unique_params / 1e9:.2f}B")

## 6. Link Shared Router

All MoE layers share the same router (gate) weights from layer 0.

In [ ]:
shared_gate = model.model.layers[0].mlp.gate
num_layers = len(model.model.layers)
for layer_idx in range(1, num_layers):
    if hasattr(model.model.layers[layer_idx].mlp, "gate"):
        model.model.layers[layer_idx].mlp.gate = shared_gate
print(f"Shared router linked across {num_layers} layers")

# Verify sharing
gate0_id = id(model.model.layers[0].mlp.gate)
all_shared = all(id(model.model.layers[i].mlp.gate) == gate0_id for i in range(1, num_layers))
print(f"All gates share same object: {all_shared}")

## 7. Training Configuration

Adjust `PER_DEVICE_BATCH_SIZE` based on your GPU VRAM:
- A100 80GB: `batch_size=8, grad_accum=64`
- A100 40GB: `batch_size=4, grad_accum=128`

If you get OOM, reduce batch size and increase grad_accum proportionally.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import math

# === ADJUST THESE BASED ON YOUR GPU ===
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 128
EFFECTIVE_BATCH = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION

OUTPUT_DIR = "./outputs/exp017_moe_shared_router_3b"
HF_PUSH_NAME = "stukenov/sozkz-moe-mix-3b-kk-base-v1"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Duration
    num_train_epochs=1,
    
    # Batch size
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    
    # Optimizer
    learning_rate=1e-4,
    weight_decay=0.1,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    warmup_steps=1000,
    
    # Precision
    bf16=True,
    
    # Logging & saving
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Performance
    dataloader_num_workers=4,
    torch_compile=True,
    report_to="none",
    
    push_to_hub=False,
)

steps_per_epoch = math.ceil(len(train_dataset) / EFFECTIVE_BATCH)
print(f"Effective batch size: {EFFECTIVE_BATCH}")
print(f"Steps per epoch: ~{steps_per_epoch:,}")
print(f"Total tokens per epoch: ~{len(train_dataset) * 1024 / 1e9:.1f}B")

## 8. Train!

In [ ]:
import os

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

# Resume from checkpoint if exists
checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        latest = max(checkpoints, key=lambda x: int(x.split("-")[1]))
        checkpoint = os.path.join(OUTPUT_DIR, latest)
        print(f"Resuming from {checkpoint}")

print("Starting training...")
train_result = trainer.train(resume_from_checkpoint=checkpoint)

## 9. Evaluate & Save

In [ ]:
results = trainer.evaluate()
print(f"Eval loss: {results['eval_loss']:.4f}")
print(f"Eval perplexity: {2 ** results['eval_loss']:.2f}")

final_dir = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Model saved to {final_dir}")

## 10. Push to HuggingFace Hub

In [ ]:
print(f"Pushing model to {HF_PUSH_NAME}...")
model.push_to_hub(HF_PUSH_NAME)
tokenizer.push_to_hub(HF_PUSH_NAME)
print("Done! Model uploaded to HuggingFace.")

## 11. Quick Test: Generate Text

In [ ]:
model.to("cuda")
prompt = "Қазақстан — "
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))